<h1 style="font-family: 'Times New Roman'; text-align: center;"><b>Machine Learning</b></h1>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b>URL Phishing Detection Project</b></h2>

<h3 style="font-family: 'Times New Roman'; text-align: right;"><b>- Pallav Biyala (24110234)</b></h3>

<hr>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b> All in One</b></h2>

### Importing Libraries

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import math
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.exceptions import ConvergenceWarning
import uuid, os, json
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Simple baselines: maximum classifier
from sklearn.dummy import DummyClassifier

## Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline # this performs regularization

## Support Vector Machines
from sklearn.svm import LinearSVC, SVC

## Decision trees, bagging and boosting
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier

## Performance metrics
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    balanced_accuracy_score
)

In [10]:
df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")

<hr>

In [11]:
import json
from pathlib import Path

folder = Path("../results/Baseline Models")

for filename in ["all_results.json", "all_confusion_matrices.json"]:
    path = folder / filename

    with open(path, "r") as f:
        records = json.load(f)

    print(f"{filename}: {len(records)} records")

all_results.json: 55 records
all_confusion_matrices.json: 55 records


<h3 style="font-family: 'Times New Roman';"><b> Pre-Processing of the Data</b></h3>

In [3]:
# Remove repeated URLs before splitting.
df = df.drop_duplicates(subset=["URL"], keep="first").copy()

# All 50 numeric input features, excluding the target.
all_numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "label"
]

print("Total numeric features:", len(all_numeric_features))

# M1: Strict lexical URL features.
lexical_features = [
    "URLLength", "DomainLength", "TLDLength", "NoOfSubDomain",
    "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL",
    "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL", "IsDomainIP",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio"
]

# M2: Add the three derived statistical features.
derived_features = [
    "TLDLegitimateProb",
    "URLCharProb",
    "CharContinuationRate"
]
# M3 adds HTTPS along with lexial and derived features --> represents what we can have if we are just given URL
# M4: Add HTTPS and the page-dependent features.
# URLSimilarityIndex is deliberately reserved for M5.
page_features = [
    c for c in all_numeric_features
    if c not in lexical_features
    and c not in derived_features
    and c not in ["IsHTTPS", "URLSimilarityIndex"]
]

feature_sets = {
    "M1 - Lexical": lexical_features,

    "M2 - Derived": (
        lexical_features + derived_features
    ),

    "M3 - URL Features": (
        lexical_features + derived_features + ["IsHTTPS"]
    ),

    "M4 - Page-aware": (
        lexical_features
        + derived_features
        + ["IsHTTPS"]
        + page_features
    ),

    "M5 - All numeric": all_numeric_features
}

for name, features in feature_sets.items():
    missing = set(features) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert len(features) == len(set(features)), (
        f"{name}: duplicate feature names"
    )
    print(f"{name}: {len(features)} features")

assert "URLSimilarityIndex" not in feature_sets["M4 - Page-aware"]
assert "URLSimilarityIndex" in feature_sets["M5 - All numeric"]

Total numeric features: 50
M1 - Lexical: 17 features
M2 - Derived: 20 features
M3 - URL Features: 21 features
M4 - Page-aware: 49 features
M5 - All numeric: 50 features


In [4]:
X = df[all_numeric_features]
y = df["label"]

# 80% train+validation, 10% test.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

# Split the remaining 90% into 80% train and 10% validation
# of the original dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=1/9,
    random_state=42,
    stratify=y_train_val
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 188296
Validation: 23537
Test: 23537


<hr>

<h3 style="font-family: 'Times New Roman';"><b> The Master Table:</b></h3>

In [5]:
RESULTS_DIR = "../results"
all_results, all_confusion_matrices = [], []
CURRENT_FOLDER = None

def start_group(folder):
    """Call at the top of each grid cell. Empties the lists and backs up old files."""
    global CURRENT_FOLDER
    all_results.clear()
    all_confusion_matrices.clear()
    CURRENT_FOLDER = folder
    path = os.path.join(RESULTS_DIR, folder)
    os.makedirs(path, exist_ok=True)
    for fname in ["all_results.json", "all_confusion_matrices.json"]:
        f = os.path.join(path, fname)
        if os.path.exists(f):
            os.replace(f, f + ".bak")      # one-level backup of the previous run
    print(f"Started '{folder}' with empty lists")

def make_json_safe(obj):
    """Convert NaN and infinity to None so the output is valid JSON."""

    if isinstance(obj, dict):
        return {key: make_json_safe(value) for key, value in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [make_json_safe(value) for value in obj]

    if isinstance(obj, (float, np.floating)):
        if not math.isfinite(float(obj)):
            return None
        return float(obj)

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    return obj


def save_group(quiet=False):
    if CURRENT_FOLDER is None:
        raise RuntimeError(
            "No active experiment group. Call start_group(folder) first."
        )

    path = os.path.join(RESULTS_DIR, CURRENT_FOLDER)
    os.makedirs(path, exist_ok=True)

    safe_results = make_json_safe(all_results)
    safe_matrices = make_json_safe(all_confusion_matrices)

    with open(
        os.path.join(path, "all_results.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            safe_results,
            f,
            indent=2,
            allow_nan=False
        )

    with open(
        os.path.join(path, "all_confusion_matrices.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            safe_matrices,
            f,
            indent=2,
            allow_nan=False
        )

    if not quiet:
        print(f"Saved {len(all_results)} records to '{CURRENT_FOLDER}'")

In [6]:
## This function adds entries to our master table.
def run_experiment(
    model,
    model_name,
    feature_set_name,
    features,
    experiment_type="Baseline",
    hyperparameters=None
):
    """
    Train a model and evaluate it on the validation set.
    Automatically append its results to the master collections.
    Records whether a ConvergenceWarning was raised during training.
    """

    # Select the appropriate features.
    X_train_selected = X_train[features]
    X_val_selected = X_val[features]

    # Train the model, measure training time, and capture warnings.
    start_time = time.time()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(X_train_selected, y_train)

    training_time = time.time() - start_time

    # Check whether a convergence warning was raised.
    convergence_warnings = [
        w for w in caught
        if issubclass(w.category, ConvergenceWarning)
    ]

    converged = len(convergence_warnings) == 0

    warning_messages = [
        str(w.message) for w in convergence_warnings
    ]

    # Generate predictions.
    y_train_pred = model.predict(X_train_selected)
    y_val_pred = model.predict(X_val_selected)

    # Calculate metrics.
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)

    balanced_acc = balanced_accuracy_score(y_val, y_val_pred)

    # Class 0 represents phishing.
    phishing_recall = recall_score(
        y_val,
        y_val_pred,
        pos_label=0
    )

    # Class order: [0 = phishing, 1 = legitimate].
    cm = confusion_matrix(
        y_val,
        y_val_pred,
        labels=[0, 1]
    )

    missed_phishing = int(cm[0, 1])
    false_alarms = int(cm[1, 0])

    # Give this specific experiment a unique identifier.
    experiment_id = str(uuid.uuid4())

    # Prepare JSON-safe model hyperparameters.
    recorded_hyperparameters = hyperparameters
    
    if isinstance(model, Pipeline):
        if recorded_hyperparameters is None or "steps" in recorded_hyperparameters:
            recorded_hyperparameters = (
                model.named_steps["classifier"].get_params()
            )
    elif recorded_hyperparameters is None:
        recorded_hyperparameters = model.get_params()
    
    # Create one standardized record.
    result = {
        "experiment_id": experiment_id,
        "model": model_name,
        "experiment_type": experiment_type,
        "feature_set": feature_set_name,
        "n_features": len(features),
        "hyperparameters": recorded_hyperparameters or {},
        "training_time_seconds": training_time,
        "train_accuracy": train_accuracy,
        "validation_accuracy": val_accuracy,
        "balanced_accuracy": balanced_acc,
        "phishing_recall": phishing_recall,
        "missed_phishing": missed_phishing,
        "false_alarms": false_alarms,
        "evaluation_split": "validation",
        "converged": converged,
        "convergence_warning_messages": warning_messages
    }

    # Add the experiment to the master collections.
    all_results.append(result)

    all_confusion_matrices.append({
        "experiment_id": experiment_id,
        "feature_set": feature_set_name,
        "confusion_matrix": cm.tolist(),
        "class_labels": {
            "0": "Phishing",
            "1": "Legitimate"
        }
    })

    save_group(quiet=True)

    # Display a compact summary.
    convergence_status = (
        "No convergence warning"
        if converged
        else "CONVERGENCE WARNING"
    )

    print(
        f"{model_name} | {feature_set_name} | "
        f"Accuracy: {val_accuracy:.4%} | "
        f"Phishing Recall: {phishing_recall:.4%} | "
        f"Missed Phishing: {missed_phishing} | "
        f"False Alarms: {false_alarms} | "
        f"Time: {training_time:.2f}s | "
        f"Convergence: {convergence_status}"
    )

    return result

<hr>

<h3 style="font-family: 'Times New Roman';"><b> The Baseline Models:</b></h3>

In [7]:
## So firstly lets create default models:
baseline_models = {
    "Majority Class": lambda: DummyClassifier(
        strategy="most_frequent"
    ),
    
    "Logistic Regression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Lasso Logistic Regression" : lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            l1_ratio=1.0,
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            tol=1e-4,
            random_state=42
        ))
    ]),

    "Ridge Logistic Regression" : lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            l1_ratio=0,
            C=1.0,
            solver="lbfgs",
            max_iter=1000,
            tol=1e-4,
            random_state=42
        ))
    ]),

    "Linear SVM": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LinearSVC(
            C=1.0,
            max_iter=3000,
            random_state=42
        ))
    ]),

    "RBF SVM": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=1.0,
            kernel="rbf",
            max_iter=-1,
            random_state=42
        ))
    ]),

    "Decision Tree": lambda: DecisionTreeClassifier(
        criterion="gini",
        random_state=42
    ),

    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": lambda: GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),

    "AdaBoost": lambda: AdaBoostClassifier(
        random_state=42
    ),

    "XGBoost": lambda: XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    )
}

print("Total baseline models:", len(baseline_models))

for name in baseline_models:
    print("-", name)

Total baseline models: 11
- Majority Class
- Logistic Regression
- Lasso Logistic Regression
- Ridge Logistic Regression
- Linear SVM
- RBF SVM
- Decision Tree
- Random Forest
- Gradient Boosting
- AdaBoost
- XGBoost


In [ ]:
start_group("Baseline Models")
total_experiments = len(baseline_models) * len(feature_sets)
completed = 0
for model_name, model_factory in baseline_models.items():

    for feature_set_name, features in feature_sets.items():

        completed += 1

        print(
            f"\n[{completed}/{total_experiments}] "
            f"{model_name} | {feature_set_name}"
        )

        model = model_factory()

        run_experiment(
            model=model,
            model_name=model_name,
            feature_set_name=feature_set_name,
            features=features,
            experiment_type="Baseline",
            hyperparameters=model.get_params()
        )
save_group()
print("\nALL BASELINE EXPERIMENTS COMPLETED!")

Started 'Baseline Models' with empty lists

[1/55] Majority Class | M1 - Lexical
Majority Class | M1 - Lexical | Accuracy: 57.2928% | Phishing Recall: 0.0000% | Missed Phishing: 10052 | False Alarms: 0 | Time: 0.01s | Convergence: No convergence warning

[2/55] Majority Class | M2 - Derived
Majority Class | M2 - Derived | Accuracy: 57.2928% | Phishing Recall: 0.0000% | Missed Phishing: 10052 | False Alarms: 0 | Time: 0.01s | Convergence: No convergence warning

[3/55] Majority Class | M3 - URL Features
Majority Class | M3 - URL Features | Accuracy: 57.2928% | Phishing Recall: 0.0000% | Missed Phishing: 10052 | False Alarms: 0 | Time: 0.01s | Convergence: No convergence warning

[4/55] Majority Class | M4 - Page-aware
Majority Class | M4 - Page-aware | Accuracy: 57.2928% | Phishing Recall: 0.0000% | Missed Phishing: 10052 | False Alarms: 0 | Time: 0.01s | Convergence: No convergence warning

[5/55] Majority Class | M5 - All numeric
Majority Class | M5 - All numeric | Accuracy: 57.2928% 

<hr>

<h3 style="font-family: 'Times New Roman';"><b> Hyperparameter Tuning:</b></h3>

<h3 style="font-family: 'Times New Roman';"><b> Ridge Logistic Regression:</b></h3>

In [8]:
start_group("Ridge Logistic Regression")
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

total_experiments = len(C_VALUES) * len(feature_sets)
completed = 0

for feature_set_name, features in feature_sets.items():
    for C_value in C_VALUES:

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                l1_ratio=0,
                C=C_value,
                solver="lbfgs",
                max_iter=1000,
                tol=1e-4,
                random_state=42
            ))
        ])

        completed += 1

        print(
            f"\n[{completed}/{total_experiments}] "
            f"Ridge Logistic Regression | "
            f"{feature_set_name} | C = {C_value}"
        )

        run_experiment(
            model=model,
            model_name="Ridge Logistic Regression",
            feature_set_name=feature_set_name,
            features=features,
            experiment_type="Hyperparameter Tuning",
            hyperparameters={
                "penalty": "l2",
                "C": C_value,
                "solver": "lbfgs",
                "max_iter": 1000
            }
        )
save_group()
print("\nRIDGE LOGISTIC REGRESSION TUNING COMPLETED!")

Started 'Ridge Logistic Regression' with empty lists

[1/35] Ridge Logistic Regression | M1 - Lexical | C = 0.001
Ridge Logistic Regression | M1 - Lexical | Accuracy: 86.9270% | Phishing Recall: 72.3140% | Missed Phishing: 2783 | False Alarms: 294 | Time: 0.33s | Convergence: No convergence warning

[2/35] Ridge Logistic Regression | M1 - Lexical | C = 0.01
Ridge Logistic Regression | M1 - Lexical | Accuracy: 87.0162% | Phishing Recall: 73.0203% | Missed Phishing: 2712 | False Alarms: 344 | Time: 0.46s | Convergence: No convergence warning

[3/35] Ridge Logistic Regression | M1 - Lexical | C = 0.1
Ridge Logistic Regression | M1 - Lexical | Accuracy: 89.1023% | Phishing Recall: 76.3231% | Missed Phishing: 2380 | False Alarms: 185 | Time: 0.72s | Convergence: No convergence warning

[4/35] Ridge Logistic Regression | M1 - Lexical | C = 1.0
Ridge Logistic Regression | M1 - Lexical | Accuracy: 97.9267% | Phishing Recall: 95.2945% | Missed Phishing: 473 | False Alarms: 15 | Time: 1.02s | Co

<h3 style="font-family: 'Times New Roman';"><b> Lasso Logistic Regression:</b></h3>

In [ ]:
start_group("Lasso Logistic Regression")
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

total_experiments = len(C_VALUES) * len(feature_sets)
completed = 0

for feature_set_name, features in feature_sets.items():

    for C_value in C_VALUES:

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                l1_ratio=1.0,
                C=C_value,
                solver="liblinear",
                max_iter=1000,
                tol=1e-4,
                random_state=42
            ))
        ])

        completed += 1

        print(
            f"\n[{completed}/{total_experiments}] "
            f"Lasso Logistic Regression | "
            f"{feature_set_name} | C = {C_value}"
        )

        run_experiment(
            model=model,
            model_name="Lasso Logistic Regression",
            feature_set_name=feature_set_name,
            features=features,
            experiment_type="Hyperparameter Tuning",
            hyperparameters={
                "penalty": "l1",
                "C": C_value,
                "solver": "liblinear",
                "max_iter": 1000,
                "tol": 1e-4
            }
        )
save_group()
print("\nLASSO LOGISTIC REGRESSION TUNING COMPLETED!")

<h3 style="font-family: 'Times New Roman';"><b> Linear SVM:</b></h3>

In [9]:
start_group("Linear SVM")
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

total_experiments = len(C_VALUES) * len(feature_sets)
completed = 0

for feature_set_name, features in feature_sets.items():
    for C_value in C_VALUES:
    
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LinearSVC(
                C=C_value,
                max_iter=3000,
                random_state=42
            ))
        ])
    
        completed += 1
    
        print(
            f"\n[{completed}/{total_experiments}] "
            f"Linear SVM | "
            f"{feature_set_name} | C = {C_value}"
        )
    
        run_experiment(
            model=model,
            model_name="Linear SVM",
            feature_set_name=feature_set_name,
            features=features,
            experiment_type="Hyperparameter Tuning",
            hyperparameters={
                "C": C_value,
                "max_iter": 3000
            }
        )

save_group()
print("\nLINEAR SVM TUNING COMPLETED!")

Started 'Linear SVM' with empty lists

[1/35] Linear SVM | M1 - Lexical | C = 0.001
Linear SVM | M1 - Lexical | Accuracy: 86.9397% | Phishing Recall: 72.2244% | Missed Phishing: 2792 | False Alarms: 282 | Time: 1.79s | Convergence: No convergence warning

[2/35] Linear SVM | M1 - Lexical | C = 0.01
Linear SVM | M1 - Lexical | Accuracy: 88.4225% | Phishing Recall: 74.5225% | Missed Phishing: 2561 | False Alarms: 164 | Time: 2.41s | Convergence: No convergence warning

[3/35] Linear SVM | M1 - Lexical | C = 0.1
Linear SVM | M1 - Lexical | Accuracy: 98.9846% | Phishing Recall: 97.6323% | Missed Phishing: 238 | False Alarms: 1 | Time: 12.36s | Convergence: No convergence warning

[4/35] Linear SVM | M1 - Lexical | C = 1.0
Linear SVM | M1 - Lexical | Accuracy: 99.5539% | Phishing Recall: 98.9554% | Missed Phishing: 105 | False Alarms: 0 | Time: 8.11s | Convergence: No convergence warning

[5/35] Linear SVM | M1 - Lexical | C = 10.0
Linear SVM | M1 - Lexical | Accuracy: 99.6856% | Phishing R

<h3 style="font-family: 'Times New Roman';"><b> RBF SVM:</b></h3>

In [ ]:
start_group("RBF SVM")
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

total_experiments = len(C_VALUES) * len(feature_sets)
completed = 0

for feature_set_name, features in feature_sets.items():
    for C_value in C_VALUES:
    
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                C=C_value,
                kernel="rbf",
                max_iter=-1,
                random_state=42
            ))
        ])
    
        completed += 1
    
        print(
            f"\n[{completed}/{total_experiments}] "
            f"RBF SVM | "
            f"{feature_set_name} | C = {C_value}"
        )
    
        run_experiment(
            model=model,
            model_name="RBF SVM",
            feature_set_name=feature_set_name,
            features=features,
            experiment_type="Hyperparameter Tuning",
            hyperparameters={
                "C": C_value,
                "kernel": "rbf",
                "max_iter": 20000
            }
        )
save_group()
print("\nRBF SVM TUNING COMPLETED!")

<h3 style="font-family: 'Times New Roman';"><b> Decision Trees:</b></h3>

In [10]:
start_group("Decision Trees")
MAX_DEPTH_VALUES = [5, 10, 15, 20, None]
MIN_SAMPLES_LEAF_VALUES = [1, 2, 5, 10, 20, 100]
CRITERION_VALUES = ["gini", "entropy"]

total_experiments = (
    len(MAX_DEPTH_VALUES)
    * len(MIN_SAMPLES_LEAF_VALUES)
    * len(CRITERION_VALUES)
    * len(feature_sets)
)

completed = 0

for feature_set_name, features in feature_sets.items():
    for max_depth_value in MAX_DEPTH_VALUES:
        for min_samples_leaf_value in MIN_SAMPLES_LEAF_VALUES:
            for criterion_value in CRITERION_VALUES:
                
                model = DecisionTreeClassifier(
                    criterion=criterion_value,
                    max_depth=max_depth_value,
                    min_samples_leaf=min_samples_leaf_value,
                    random_state=42
                )
    
                completed += 1
    
                print(
                    f"\n[{completed}/{total_experiments}] "
                    f"Decision Tree | {feature_set_name} | "
                    f"max_depth={max_depth_value} | "
                    f"min_samples_leaf={min_samples_leaf_value} | "
                    f"criterion={criterion_value}"
                )
    
                run_experiment(
                    model=model,
                    model_name="Decision Tree",
                    feature_set_name=feature_set_name,
                    features=features,
                    experiment_type="Hyperparameter Tuning",
                    hyperparameters={
                        "max_depth": max_depth_value,
                        "min_samples_leaf": min_samples_leaf_value,
                        "criterion": criterion_value
                    }
                )
    
save_group()
print("\nDECISION TREE HYPERPARAMETER TUNING COMPLETED!")

Started 'Decision Trees' with empty lists

[1/300] Decision Tree | M1 - Lexical | max_depth=5 | min_samples_leaf=1 | criterion=gini
Decision Tree | M1 - Lexical | Accuracy: 86.1027% | Phishing Recall: 70.7322% | Missed Phishing: 2942 | False Alarms: 329 | Time: 0.21s | Convergence: No convergence warning

[2/300] Decision Tree | M1 - Lexical | max_depth=5 | min_samples_leaf=1 | criterion=entropy
Decision Tree | M1 - Lexical | Accuracy: 85.7968% | Phishing Recall: 68.6033% | Missed Phishing: 3156 | False Alarms: 187 | Time: 0.19s | Convergence: No convergence warning

[3/300] Decision Tree | M1 - Lexical | max_depth=5 | min_samples_leaf=2 | criterion=gini
Decision Tree | M1 - Lexical | Accuracy: 86.1027% | Phishing Recall: 70.7322% | Missed Phishing: 2942 | False Alarms: 329 | Time: 0.19s | Convergence: No convergence warning

[4/300] Decision Tree | M1 - Lexical | max_depth=5 | min_samples_leaf=2 | criterion=entropy
Decision Tree | M1 - Lexical | Accuracy: 85.7968% | Phishing Recall: 6

<h3 style="font-family: 'Times New Roman';"><b> Random Forests:</b></h3>

In [11]:
start_group("Random Forests")
N_ESTIMATORS_VALUES = [100, 200]
MAX_DEPTH_VALUES = [10, 20, None]
MIN_SAMPLES_LEAF_VALUES = [1, 5, 10]

total_experiments = (
    len(N_ESTIMATORS_VALUES)
    * len(MAX_DEPTH_VALUES)
    * len(MIN_SAMPLES_LEAF_VALUES)
    * len(feature_sets)
)

completed = 0

for feature_set_name, features in feature_sets.items():
    for n_estimators_value in N_ESTIMATORS_VALUES:
        for max_depth_value in MAX_DEPTH_VALUES:
            for min_samples_leaf_value in MIN_SAMPLES_LEAF_VALUES:
    
                model = RandomForestClassifier(
                    n_estimators=n_estimators_value,
                    max_depth=max_depth_value,
                    min_samples_leaf=min_samples_leaf_value,
                    criterion="gini",
                    max_features="sqrt",
                    random_state=42,
                    n_jobs=-1
                )
    
                completed += 1
    
                print(
                    f"\n[{completed}/{total_experiments}] "
                    f"Random Forest | {feature_set_name} | "
                    f"n_estimators={n_estimators_value} | "
                    f"max_depth={max_depth_value} | "
                    f"min_samples_leaf={min_samples_leaf_value} | "
                )
    
                run_experiment(
                    model=model,
                    model_name="Random Forest",
                    feature_set_name=feature_set_name,
                    features=features,
                    experiment_type="Hyperparameter Tuning",
                    hyperparameters={
                        "n_estimators": n_estimators_value,
                        "max_depth": max_depth_value,
                        "min_samples_leaf": min_samples_leaf_value,
                        "criterion": "gini",
                        "max_features": "sqrt"
                    }
                )

save_group()
print("\nRANDOM FOREST HYPERPARAMETER TUNING COMPLETED!")

Started 'Random Forests' with empty lists

[1/90] Random Forest | M1 - Lexical | n_estimators=100 | max_depth=10 | min_samples_leaf=1 | 
Random Forest | M1 - Lexical | Accuracy: 91.4475% | Phishing Recall: 80.8993% | Missed Phishing: 1920 | False Alarms: 93 | Time: 4.52s | Convergence: No convergence warning

[2/90] Random Forest | M1 - Lexical | n_estimators=100 | max_depth=10 | min_samples_leaf=5 | 
Random Forest | M1 - Lexical | Accuracy: 91.3031% | Phishing Recall: 80.6009% | Missed Phishing: 1950 | False Alarms: 97 | Time: 4.82s | Convergence: No convergence warning

[3/90] Random Forest | M1 - Lexical | n_estimators=100 | max_depth=10 | min_samples_leaf=10 | 
Random Forest | M1 - Lexical | Accuracy: 91.1246% | Phishing Recall: 80.2626% | Missed Phishing: 1984 | False Alarms: 105 | Time: 7.32s | Convergence: No convergence warning

[4/90] Random Forest | M1 - Lexical | n_estimators=100 | max_depth=20 | min_samples_leaf=1 | 
Random Forest | M1 - Lexical | Accuracy: 99.6686% | Phish

<h3 style="font-family: 'Times New Roman';"><b> Gradient Boosting:</b></h3>

In [12]:
start_group("Gradient Boosting")
N_ESTIMATORS_VALUES = [50, 100, 200]
LEARNING_RATE_VALUES = [0.05, 0.1, 0.2]
MAX_DEPTH_VALUES = [2, 5, 7]

total_experiments = (
    len(N_ESTIMATORS_VALUES)
    * len(LEARNING_RATE_VALUES)
    * len(MAX_DEPTH_VALUES)
    * len(feature_sets)
)

completed = 0

for feature_set_name, features in feature_sets.items():
    for n_estimators_value in N_ESTIMATORS_VALUES:
        for learning_rate_value in LEARNING_RATE_VALUES:
            for max_depth_value in MAX_DEPTH_VALUES:
    
                model = GradientBoostingClassifier(
                    n_estimators=n_estimators_value,
                    learning_rate=learning_rate_value,
                    max_depth=max_depth_value,
                    random_state=42
                )
    
                completed += 1
    
                print(
                    f"\n[{completed}/{total_experiments}] "
                    f"Gradient Boosting | {feature_set_name} | "
                    f"n_estimators={n_estimators_value} | "
                    f"learning_rate={learning_rate_value} | "
                    f"max_depth={max_depth_value}"
                )
    
                run_experiment(
                    model=model,
                    model_name="Gradient Boosting",
                    feature_set_name=feature_set_name,
                    features=features,
                    experiment_type="Hyperparameter Tuning",
                    hyperparameters={
                        "n_estimators": n_estimators_value,
                        "learning_rate": learning_rate_value,
                        "max_depth": max_depth_value
                    }
                )

save_group()
print("\nGRADIENT BOOSTING TUNING COMPLETED!")

Started 'Gradient Boosting' with empty lists

[1/135] Gradient Boosting | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=2
Gradient Boosting | M1 - Lexical | Accuracy: 85.3422% | Phishing Recall: 71.4186% | Missed Phishing: 2873 | False Alarms: 577 | Time: 5.43s | Convergence: No convergence warning

[2/135] Gradient Boosting | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=5
Gradient Boosting | M1 - Lexical | Accuracy: 94.7317% | Phishing Recall: 89.2758% | Missed Phishing: 1078 | False Alarms: 162 | Time: 11.61s | Convergence: No convergence warning

[3/135] Gradient Boosting | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=7
Gradient Boosting | M1 - Lexical | Accuracy: 98.2793% | Phishing Recall: 96.8464% | Missed Phishing: 317 | False Alarms: 88 | Time: 20.70s | Convergence: No convergence warning

[4/135] Gradient Boosting | M1 - Lexical | n_estimators=50 | learning_rate=0.1 | max_depth=2
Gradient Boosting | M1 - Lexical | Accurac

<h3 style="font-family: 'Times New Roman';"><b> Ada Boosting:</b></h3>

In [13]:
start_group("Ada Boosting")
N_ESTIMATORS_VALUES = [50, 100, 200]
LEARNING_RATE_VALUES = [0.05, 0.1, 0.2]

total_experiments = (
    len(N_ESTIMATORS_VALUES)
    * len(LEARNING_RATE_VALUES)
    * len(feature_sets)
)

completed = 0

for feature_set_name, features in feature_sets.items():
    for n_estimators_value in N_ESTIMATORS_VALUES:
        for learning_rate_value in LEARNING_RATE_VALUES:
    
            model = AdaBoostClassifier(
                n_estimators=n_estimators_value,
                learning_rate=learning_rate_value,
                random_state=42
            )
    
            completed += 1
    
            print(
                f"\n[{completed}/{total_experiments}] "
                f"AdaBoost | {feature_set_name} | "
                f"n_estimators={n_estimators_value} | "
                f"learning_rate={learning_rate_value}"
            )
    
            run_experiment(
                model=model,
                model_name="AdaBoost",
                feature_set_name=feature_set_name,
                features=features,
                experiment_type="Hyperparameter Tuning",
                hyperparameters={
                    "n_estimators": n_estimators_value,
                    "learning_rate": learning_rate_value
                }
            )

save_group()
print("\nADABOOST TUNING COMPLETED!")

Started 'Ada Boosting' with empty lists

[1/45] AdaBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.05
AdaBoost | M1 - Lexical | Accuracy: 80.4818% | Phishing Recall: 57.9984% | Missed Phishing: 4222 | False Alarms: 372 | Time: 5.40s | Convergence: No convergence warning

[2/45] AdaBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.1
AdaBoost | M1 - Lexical | Accuracy: 82.3384% | Phishing Recall: 62.9129% | Missed Phishing: 3728 | False Alarms: 429 | Time: 4.55s | Convergence: No convergence warning

[3/45] AdaBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.2
AdaBoost | M1 - Lexical | Accuracy: 82.5721% | Phishing Recall: 64.6041% | Missed Phishing: 3558 | False Alarms: 544 | Time: 4.63s | Convergence: No convergence warning

[4/45] AdaBoost | M1 - Lexical | n_estimators=100 | learning_rate=0.05
AdaBoost | M1 - Lexical | Accuracy: 82.3384% | Phishing Recall: 62.9129% | Missed Phishing: 3728 | False Alarms: 429 | Time: 10.13s | Convergence: No convergence warning


<h3 style="font-family: 'Times New Roman';"><b> XG Boosting:</b></h3>

In [14]:
start_group("XG Boosting")
N_ESTIMATORS_VALUES = [50, 100, 200]
LEARNING_RATE_VALUES = [0.05, 0.1, 0.2]
MAX_DEPTH_VALUES = [2, 5, 7]

total_experiments = (
    len(N_ESTIMATORS_VALUES)
    * len(LEARNING_RATE_VALUES)
    * len(MAX_DEPTH_VALUES)
    * len(feature_sets)
)

completed = 0

for feature_set_name, features in feature_sets.items():
    for n_estimators_value in N_ESTIMATORS_VALUES:
        for learning_rate_value in LEARNING_RATE_VALUES:
            for max_depth_value in MAX_DEPTH_VALUES:

                model = XGBClassifier(
                    objective="binary:logistic",
                    n_estimators=n_estimators_value,
                    learning_rate=learning_rate_value,
                    max_depth=max_depth_value,
                    subsample=1.0,
                    random_state=42,
                    n_jobs=-1,
                    eval_metric="logloss"
                )

                completed += 1

                print(
                    f"\n[{completed}/{total_experiments}] "
                    f"XGBoost | {feature_set_name} | "
                    f"n_estimators={n_estimators_value} | "
                    f"learning_rate={learning_rate_value} | "
                    f"max_depth={max_depth_value} | "
                )

                run_experiment(
                    model=model,
                    model_name="XGBoost",
                    feature_set_name=feature_set_name,
                    features=features,
                    experiment_type="Hyperparameter Tuning",
                    hyperparameters={
                        "n_estimators": n_estimators_value,
                        "learning_rate": learning_rate_value,
                        "max_depth": max_depth_value,
                        "subsample": 1.0
                    }
                )
save_group()
print("\nXGBOOST TUNING COMPLETED!")

Started 'XG Boosting' with empty lists

[1/135] XGBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=2 | 
XGBoost | M1 - Lexical | Accuracy: 85.1086% | Phishing Recall: 69.3394% | Missed Phishing: 3082 | False Alarms: 423 | Time: 1.11s | Convergence: No convergence warning

[2/135] XGBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=5 | 
XGBoost | M1 - Lexical | Accuracy: 92.6754% | Phishing Recall: 84.0629% | Missed Phishing: 1602 | False Alarms: 122 | Time: 0.45s | Convergence: No convergence warning

[3/135] XGBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.05 | max_depth=7 | 
XGBoost | M1 - Lexical | Accuracy: 97.9522% | Phishing Recall: 96.0605% | Missed Phishing: 396 | False Alarms: 86 | Time: 0.54s | Convergence: No convergence warning

[4/135] XGBoost | M1 - Lexical | n_estimators=50 | learning_rate=0.1 | max_depth=2 | 
XGBoost | M1 - Lexical | Accuracy: 86.0857% | Phishing Recall: 71.2396% | Missed Phishing: 2891 | False Alar

<hr>